# 02 — Prepare Labels
Merge ECOSoundSet and InsectSet459, balance classes, and produce the `train.csv` / `val.csv` / `test.csv` splits that OpenSoundscape expects.

**Kernel:** `Python (orthoptera-training)`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import soundfile as sf
import librosa

PROJECT_ROOT = Path.cwd().parent.parent  # cwd relative to current notebook path
ECO_ROOT    = PROJECT_ROOT / "datasets" / "ecosoundset"
INSECT_ROOT = PROJECT_ROOT / "datasets" / "insectset459"
OUT_DIR     = PROJECT_ROOT / "training" / "data"
OUT_DIR.mkdir(exist_ok=True)

# Canonical species names as they appear in ECOSoundSet (trinomial).
# InsectSet459 uses underscore format — mapped below.
# Meconema thalassinum is absent from both datasets.
UK_SPECIES = [
    "Chorthippus brunneus brunneus",          # Field Grasshopper
    "Pseudochorthippus parallelus parallelus", # Meadow Grasshopper
    "Omocestus viridulus",                    # Common Green Grasshopper
    "Tettigonia viridissima",                 # Great Green Bush-cricket
    "Roeseliana roeselii",                    # Roesel's Bush-cricket
    "Pholidoptera griseoaptera",              # Dark Bush-cricket
    "Leptophyes punctatissima",               # Speckled Bush-cricket
    "Gryllus campestris",                     # Field Cricket
]

# InsectSet459 species_name uses underscores and may be binomial.
# Map species_name from InsectSet459 to ECO canonical name.
INSECT_TO_ECO = {
    "Chorthippus_brunneus":                   "Chorthippus brunneus brunneus",
    "Pseudochorthippus_parallelus":           "Pseudochorthippus parallelus parallelus",
    "Omocestus_viridulus":                    "Omocestus viridulus",
    "Tettigonia_viridissima":                 "Tettigonia viridissima",
    "Roeseliana_roeselii":                    "Roeseliana roeselii",
    "Pholidoptera_griseoaptera":              "Pholidoptera griseoaptera",
    "Leptophyes_punctatissima":               "Leptophyes punctatissima",
    "Gryllus_campestris":                     "Gryllus campestris"
}


In [ ]:
# ── Load ECOSoundSet and split by subset column ───────────────────────────────
all_annot_eco = pd.read_csv(ECO_ROOT / "annotated_audio_segments.csv")

# Search recursively within entire ecosoundset directory to build file index.
# Removes need to assume directory/subdirectory structure.
eco_file_index = {file_path.name: file_path for file_path in ECO_ROOT.rglob("*.wav")}
all_annot_eco["filepath"] = all_annot_eco["audio_segment_file_name"].map(eco_file_index)
all_annot_eco = all_annot_eco[~all_annot_eco["filepath"].isna()]

all_annot_eco.rename(columns={
    "label": "species",
    "audio_segment_initial_time": "t_min",
    "audio_segment_final_time":   "t_max",
}, inplace=True)

# Every EcoSoundSet clip is 4 seconds, t_min and t_max must be relative to the split clip, not the whole clip.
# We take the actual duration as some clips are slightly less than 4s, causing noisy errors.
all_annot_eco["t_max"] = all_annot_eco["filepath"].apply(lambda fp: sf.info(fp).duration).clip(upper=4.0)
all_annot_eco["t_min"] = 0.0

# Filter for both target orthoptera annotations and clips with no orthoptera annotations to provide model context for non-Orthoptera sounds.
orth_eco = all_annot_eco[
    (all_annot_eco["label_category"] == "Orthoptera") &
    (all_annot_eco["species"].isin(UK_SPECIES))
].copy()

neg_orth_eco = all_annot_eco[
    (all_annot_eco["label_category"] != "Orthoptera") &
    (~all_annot_eco["filepath"].isin(orth_eco["filepath"]))
].copy()

# Only take 40% of the negative training samples to keep the "no species" class in balance with other species.
# 40% of the negative training samples leaves us with 3,853 samples, in line with the 3,500-4,000 class average after resampling in 03_train_model.
neg_train_sample = neg_orth_eco[neg_orth_eco["subset"] == "train"].sample(frac=0.40, random_state=42)
neg_val = neg_orth_eco[neg_orth_eco["subset"] == "val"]#.sample(frac=0.20, random_state=42)
neg_test = neg_orth_eco[neg_orth_eco["subset"] == "test"]#.sample(frac=0.15, random_state=42)
print(neg_val["filepath"].nunique())
print(neg_test["filepath"].nunique())

eco_train = pd.concat([orth_eco[orth_eco["subset"] == "train"], neg_train_sample], ignore_index=True)
eco_val   = pd.concat([orth_eco[orth_eco["subset"] == "val"], neg_val], ignore_index=True)
eco_test  = pd.concat([orth_eco[orth_eco["subset"] == "test"], neg_test], ignore_index=True)
print(f"ECOSoundSet annotations — train: {len(eco_train)}  val: {len(eco_val)}  test: {len(eco_test)}")
print(f"ECOSoundSet clips (w/o neg) — train: {orth_eco[orth_eco['subset'] == 'train']['filepath'].nunique()}  val: {orth_eco[orth_eco['subset'] == 'val']['filepath'].nunique()}  test: {orth_eco[orth_eco['subset'] == 'test']['filepath'].nunique()}")
print(f"ECOSoundSet clips — train: {eco_train['filepath'].nunique()}  val: {eco_val['filepath'].nunique()}  test: {eco_test['filepath'].nunique()}")
#print(f"EcoSoundSet clips per species: {orth_eco[orth_eco['subset'] == 'val'].groupby('species')['filepath'].nunique()}")
eco_train["species"].value_counts()


In [ ]:
# ── Supplement with InsectSet459 (train and validation splits) ──────────────────────────
# InsectSet459 has no test split — we use it only to top up training data.
all_annot_insect = pd.read_csv(INSECT_ROOT / "InsectSet459_Train_Val_Annotation.csv")

# Map matching species_names to canonical counterpart in INSECT_TO_ECO.
# .map() returns NaN for any key (species_name) not in INSECT_TO_ECO.
all_annot_insect["species"] = all_annot_insect["species_name"].map(INSECT_TO_ECO)
insect_train = all_annot_insect[all_annot_insect["species"].notna()].copy()

# Search recursively within entire insectset459 directory to build file index.
# Removes need to assume directory/subdirectory structure.
insect_file_index = {f.name: f for f in INSECT_ROOT.rglob("*") if f.suffix in (".wav", ".mp3")}
insect_train["filepath"] = insect_train["file_name"].map(insect_file_index)

insect_train["t_min"] = 0.0
insect_train["t_max"] = insect_train["filepath"].apply(lambda fp: sf.info(fp).duration)

def find_best_4s_window(filepath, window_duration=4.0):
    y, sr = librosa.load(filepath, sr=None)
    total_duration = len(y) / sr
    
    hop_length = 512
    n_mels = 128
    fmax = sr / 2 # File's Nyquist frequency.
    
    S = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        hop_length=hop_length,
        n_mels=n_mels,
        fmax=fmax,
    )
    
    mel_freqs = librosa.mel_frequencies(
        n_mels=n_mels,
        fmax=fmax,
    )
    
    energy = S[mel_freqs >= 2000].sum(axis=0)
    
    # Find the time of peak energy using a sliding window
    frames_per_window = int(window_duration * sr / hop_length)

    # If clip is shorter than 4s, we can't find the best 4s clip.
    if len(energy) < frames_per_window:
        return 0.0, total_duration
    
    # Convolve with a rectangular window to find highest-energy 4s region
    window_energy = np.convolve(energy, np.ones(frames_per_window), mode='valid')
    best_frame = window_energy.argmax()
    
    t_min = librosa.frames_to_time(best_frame, sr=sr, hop_length=hop_length)
    t_max = min(t_min + window_duration, total_duration)
    t_min = max(0, t_max - window_duration)
    
    return t_min, t_max

insect_train[["t_min", "t_max"]] = insect_train["filepath"].apply(
    lambda fp: pd.Series(find_best_4s_window(fp))
)

print(f"InsectSet459 — train (train + validation subsets): {len(insect_train)}")
insect_train["species"].value_counts()


In [ ]:
# ── Merge and write OpenSoundscape one-hot CSVs ──────────────────────────────
# OSS expects index (file, start_time, end_time) + one-hot species columns.

def to_oss_format(df):
    # Group by clip to handle multiple species annotations per clip (relevant for ECOSoundSet).
    # Ensures multi-hot encoded cols indicate all species that appear in the clip.
    grouped = df.groupby(["filepath", "t_min", "t_max"])["species"].apply(set).reset_index()
    
    rows = []
    for _, row in grouped.iterrows():
        entry = {
            "file":       row["filepath"],
            "start_time": row["t_min"],
            "end_time":   row["t_max"],
        }
        for sp in UK_SPECIES:
            entry[sp] = 1 if sp in row["species"] else 0
        rows.append(entry)
    return pd.DataFrame(rows).set_index(["file", "start_time", "end_time"])

# Combine ECO + InsectSet459 for training; use ECO only for val/test
combined_train = pd.concat([eco_train, insect_train], ignore_index=True)

train_oss = to_oss_format(combined_train)
val_oss   = to_oss_format(eco_val)
test_oss  = to_oss_format(eco_test)

train_oss.to_csv(OUT_DIR / "train.csv")
val_oss.to_csv(OUT_DIR / "val.csv")
test_oss.to_csv(OUT_DIR / "test.csv")

print(f"Saved to {OUT_DIR}/")
print(f"Train: {len(train_oss)}  Val: {len(val_oss)}  Test: {len(test_oss)}")
print("\nPer-species train counts:")
print(train_oss.sum().sort_values(ascending=False).to_string())


In [ ]:
print(val_oss.sum())